# Match a provider's products to existing assets

Generalized version of `coinbase-asset-match/match-coinbase-assets.ipynb`. Set `PROVIDER_NAME` below and run all cells. Currently supports `Coinbase`, `Kraken`, and `OKX`. To add a new provider, write a `ProviderConnector` subclass and register it in `CONNECTORS`.

**Inputs:** `POSTGRES_URL` from `.env`. Provider APIs are public (no auth needed).

**Outputs (named `*_<provider>.csv`):**
1. `review_<provider>.csv` — one row per unique provider currency code with the best local-asset match, the match strategy, the provider's name + description (so you can sanity-check matches), example product ids, and `asset_provider_codes` showing how the matched local asset is coded by every other provider already in the DB.
2. `local_assets_no_<provider>.csv` — active local assets whose symbol does not appear in any of the provider's products.
3. `price_check_<provider>.csv` — for each matched asset that has a USD-or-USDT pair on this provider, compares this provider's current price to the latest USD close from a reference provider (Kraken by default). Use this to spot bad matches: a >5% gap usually means a ticker collision.
4. `provider_asset` rows in Postgres — the final cell upserts directly via SQLAlchemy (only `exact_*` and `ci_*` matches; fuzzy stays manual).

In [1]:
# === CONFIG: change this one variable to switch providers ===
PROVIDER_NAME = "OKX"   # "Coinbase" | "Kraken" | "OKX"
REFERENCE_PROVIDER_NAME = "Kraken"   # used for the price-check sanity test
FUZZY_THRESHOLD = 0.85
PRICE_DIFF_PCT_FLAG = 5.0   # surface matches that disagree by more than this
REFERENCE_LOOKBACK_DAYS = 7

In [2]:
import os
import difflib
import datetime as dt
from abc import ABC, abstractmethod
from collections import Counter, defaultdict

import pandas as pd
import requests
from dotenv import load_dotenv
from sqlalchemy import create_engine, select
from sqlalchemy.dialects.postgresql import insert as pg_insert

import mc_postgres_db.models as models

load_dotenv()
POSTGRES_URL = os.getenv("POSTGRES_URL")
engine = create_engine(POSTGRES_URL)

## Provider connectors

Each connector returns two DataFrames:

- `fetch_currencies()` — columns: `code`, `name`, `description`, `type`. One row per unique currency code the provider knows about. `name`/`description` are used for fuzzy matching and human review.
- `fetch_products()` — columns: `product_id`, `base_code`, `quote_code`, `raw_status`, `is_live`. One row per trading pair. `raw_status` is the provider's raw status string (passed through to the review CSV); `is_live` is the connector's own decision about whether the pair is currently tradable (different providers use different status fields).

In [3]:
class ProviderConnector(ABC):
    name: str

    @abstractmethod
    def fetch_currencies(self) -> pd.DataFrame: ...

    @abstractmethod
    def fetch_products(self) -> pd.DataFrame: ...


class CoinbaseConnector(ProviderConnector):
    name = "Coinbase"
    PRODUCTS_URL = "https://api.coinbase.com/api/v3/brokerage/market/products"
    CURRENCIES_URL = "https://api.exchange.coinbase.com/currencies"

    def fetch_currencies(self) -> pd.DataFrame:
        resp = requests.get(self.CURRENCIES_URL, timeout=30)
        resp.raise_for_status()
        out = []
        for c in resp.json():
            details = c.get("details") or {}
            out.append(
                {
                    "code": c.get("id"),
                    "name": c.get("name"),
                    "description": details.get("description"),
                    "type": details.get("type"),
                }
            )
        return pd.DataFrame(out).drop_duplicates(subset="code")

    def fetch_products(self) -> pd.DataFrame:
        resp = requests.get(self.PRODUCTS_URL, timeout=30)
        resp.raise_for_status()
        df = pd.DataFrame(resp.json().get("products", []))

        def _bool(s: pd.Series) -> pd.Series:
            return s.fillna(False).astype(bool)

        is_live = (
            (df["status"] == "online")
            & ~_bool(df.get("is_disabled", False))
            & ~_bool(df.get("trading_disabled", False))
            & ~_bool(df.get("view_only", False))
            & ~_bool(df.get("auction_mode", False))
        )
        return pd.DataFrame(
            {
                "product_id": df["product_id"],
                "base_code": df["base_currency_id"],
                "quote_code": df["quote_currency_id"],
                "raw_status": df["status"],
                "is_live": is_live,
                "price": df.get("price"),
            }
        )


class KrakenConnector(ProviderConnector):
    name = "Kraken"
    ASSET_PAIRS_URL = "https://api.kraken.com/0/public/AssetPairs"
    ASSETS_URL = "https://api.kraken.com/0/public/Assets"

    def fetch_currencies(self) -> pd.DataFrame:
        resp = requests.get(self.ASSETS_URL, timeout=30)
        resp.raise_for_status()
        body = resp.json()
        if body.get("error"):
            raise RuntimeError(f"Kraken errors: {body['error']}")
        out = []
        for code, info in body["result"].items():
            out.append(
                {
                    "code": code,
                    "name": info.get("altname") or code,
                    "description": None,
                    "type": info.get("aclass"),
                }
            )
        return pd.DataFrame(out).drop_duplicates(subset="code")

    def fetch_products(self) -> pd.DataFrame:
        resp = requests.get(self.ASSET_PAIRS_URL, timeout=30)
        resp.raise_for_status()
        body = resp.json()
        if body.get("error"):
            raise RuntimeError(f"Kraken errors: {body['error']}")
        out = []
        for code, info in body["result"].items():
            venue = info.get("execution_venue", "international")
            out.append(
                {
                    "product_id": code,
                    "base_code": info.get("base"),
                    "quote_code": info.get("quote"),
                    "raw_status": venue,
                    "is_live": venue == "international",
                }
            )
        return pd.DataFrame(out)


class OKXConnector(ProviderConnector):
    name = "OKX"
    INSTRUMENTS_URL = "https://www.okx.com/api/v5/public/instruments"
    TICKERS_URL = "https://www.okx.com/api/v5/market/tickers"

    def fetch_currencies(self) -> pd.DataFrame:
        # OKX has no public currency-metadata endpoint; derive code-only rows
        # from the union of base/quote codes seen in products. `name` falls
        # back to the code itself, so fuzzy matching still works on the code.
        prods = self.fetch_products()
        codes = sorted(set(prods["base_code"]) | set(prods["quote_code"]))
        return pd.DataFrame(
            [
                {"code": c, "name": c, "description": None, "type": "crypto"}
                for c in codes
                if isinstance(c, str)
            ]
        )

    def fetch_products(self) -> pd.DataFrame:
        resp = requests.get(self.INSTRUMENTS_URL, params={"instType": "SPOT"}, timeout=30)
        resp.raise_for_status()
        body = resp.json()
        if str(body.get("code")) != "0":
            raise RuntimeError(f"OKX code={body.get('code')} msg={body.get('msg')}")
        df = pd.DataFrame(body["data"])

        # OKX's instruments endpoint omits price. Pull all SPOT tickers in one
        # call and merge `last` -> `price` so the price-check cell can run.
        t = requests.get(self.TICKERS_URL, params={"instType": "SPOT"}, timeout=30)
        t.raise_for_status()
        tbody = t.json()
        if str(tbody.get("code")) != "0":
            raise RuntimeError(f"OKX code={tbody.get('code')} msg={tbody.get('msg')}")
        tickers = pd.DataFrame(tbody["data"])[["instId", "last"]]

        merged = df.merge(tickers, on="instId", how="left")
        return pd.DataFrame(
            {
                "product_id": merged["instId"],
                "base_code": merged["baseCcy"],
                "quote_code": merged["quoteCcy"],
                "raw_status": merged["state"],
                "is_live": merged["state"] == "live",
                "price": merged["last"],
            }
        )


CONNECTORS: dict[str, ProviderConnector] = {
    "Coinbase": CoinbaseConnector(),
    "Kraken": KrakenConnector(),
    "OKX": OKXConnector(),
}
connector = CONNECTORS[PROVIDER_NAME]
print(f"Using connector: {connector.name}")

Using connector: OKX


## Load local assets and existing provider mappings

Pulls the canonical `asset` table plus every active `provider_asset` row, so we can show the cross-provider codes ("this asset is `Kraken:XXBT`, `Coinbase:BTC`") for each match in the review CSV.

In [4]:
asset_df = pd.read_sql(
    select(
        models.Asset.id,
        models.Asset.symbol,
        models.Asset.name,
        models.Asset.description,
        models.Asset.is_active,
    ),
    engine,
)

provider_df = pd.read_sql(
    select(models.Provider.id, models.Provider.name),
    engine,
)

this_provider = provider_df[provider_df["name"] == PROVIDER_NAME]
if this_provider.empty:
    raise RuntimeError(
        f"Provider {PROVIDER_NAME!r} not found in DB. Insert the Provider "
        "row first (see Part B in the plan)."
    )
this_provider_id = int(this_provider["id"].iloc[0])

all_provider_assets = pd.read_sql(
    select(
        models.ProviderAsset.provider_id,
        models.ProviderAsset.asset_id,
        models.ProviderAsset.asset_code,
        models.ProviderAsset.date,
        models.ProviderAsset.is_active,
    ).where(models.ProviderAsset.is_active == True),
    engine,
)
latest_provider_assets = (
    all_provider_assets.sort_values("date")
    .drop_duplicates(["asset_id", "provider_id"], keep="last")
    .merge(
        provider_df.rename(columns={"id": "provider_id", "name": "provider_name"}),
        on="provider_id",
        how="left",
    )
)

provider_codes_by_asset: dict[int, str] = {}
for aid, group in latest_provider_assets.groupby("asset_id"):
    rows = group.sort_values("provider_name")
    provider_codes_by_asset[int(aid)] = ", ".join(
        f"{r['provider_name']}:{r['asset_code']}" for _, r in rows.iterrows()
    )

asset_description_by_id: dict[int, str] = {
    int(r["id"]): r["description"]
    for _, r in asset_df.iterrows()
    if pd.notna(r["description"])
}

existing_mappings = (
    all_provider_assets[all_provider_assets["provider_id"] == this_provider_id][
        ["asset_code", "asset_id", "date", "is_active"]
    ].copy()
)

print(f"Local assets: {len(asset_df):,} ({asset_df['is_active'].sum()} active)")
print(f"{PROVIDER_NAME} provider id: {this_provider_id}")
print(f"Existing {PROVIDER_NAME} provider_asset rows: {len(existing_mappings):,}")
print(
    f"All providers: {latest_provider_assets['provider_name'].nunique()} distinct, "
    f"{len(latest_provider_assets):,} active (asset, provider) rows"
)

Local assets: 86 (83 active)
OKX provider id: 57
Existing OKX provider_asset rows: 0
All providers: 3 distinct, 161 active (asset, provider) rows


## Fetch the provider's products and currencies

In [5]:
products_df = connector.fetch_products()
print(f"{PROVIDER_NAME} returned {len(products_df):,} products")
live_df = products_df[products_df["is_live"]].copy()
print(f"Live products: {len(live_df):,} / {len(products_df):,}")
live_df.head()

OKX returned 1,237 products
Live products: 1,237 / 1,237


,product_id,base_code,quote_code,raw_status,is_live,price
0,USDT-SGD,USDT,SGD,live,True,1.2752
1,USDC-SGD,USDC,SGD,live,True,1.2727
2,BTC-AUD,BTC,AUD,live,True,109105.8
3,ETH-AUD,ETH,AUD,live,True,3220
4,SOL-AUD,SOL,AUD,live,True,116.97


In [6]:
currencies_df = connector.fetch_currencies()
print(f"{PROVIDER_NAME} returned metadata for {len(currencies_df):,} currencies")
currencies_df.head()

OKX returned metadata for 305 currencies


,code,name,description,type
0,1INCH,1INCH,None,crypto
1,2Z,2Z,None,crypto
2,A,A,None,crypto
3,AAVE,AAVE,None,crypto
4,ACE,ACE,None,crypto


## Collect unique currency codes used in live products

We care about the codes that appear in `base_code` / `quote_code` — those are the strings that need to land in `provider_asset.asset_code`. We also keep counts and example product ids per code.

In [7]:
base_counts: Counter[str] = Counter()
quote_counts: Counter[str] = Counter()
examples: dict[str, list[str]] = defaultdict(list)

for _, row in live_df.iterrows():
    base = row["base_code"]
    quote = row["quote_code"]
    pid = row["product_id"]
    if isinstance(base, str):
        base_counts[base] += 1
        if len(examples[base]) < 3:
            examples[base].append(pid)
    if isinstance(quote, str):
        quote_counts[quote] += 1
        if len(examples[quote]) < 3:
            examples[quote].append(pid)

codes_df = pd.DataFrame(
    [
        {
            "code": code,
            "base_count": base_counts.get(code, 0),
            "quote_count": quote_counts.get(code, 0),
            "total_products": base_counts.get(code, 0) + quote_counts.get(code, 0),
            "example_products": ", ".join(examples[code]),
        }
        for code in sorted(set(base_counts) | set(quote_counts))
    ]
).sort_values("total_products", ascending=False)

codes_df = codes_df.merge(
    currencies_df.rename(
        columns={
            "name": "provider_name",
            "description": "provider_description",
            "type": "provider_type",
        }
    ),
    on="code",
    how="left",
)

missing_meta = codes_df["provider_name"].isna().sum()
print(
    f"{len(codes_df):,} unique {PROVIDER_NAME} currency codes "
    f"({missing_meta} without metadata)"
)
codes_df.head(15)

305 unique OKX currency codes (0 without metadata)


,code,base_count,quote_count,total_products,example_products,provider_name,provider_description,provider_type
0,USDT,7,296,303,"USDT-SGD, USDT-AUD, USDT-AED",USDT,None,crypto
1,USD,0,284,284,"BTC-USD, ETH-USD, SOL-USD",USD,None,crypto
2,USDC,6,254,260,"USDC-SGD, USDC-AUD, USDC-BRL",USDC,None,crypto
3,EUR,0,257,257,"BTC-EUR, ETH-EUR, SOL-EUR",EUR,None,crypto
4,TRY,0,114,114,"BTC-TRY, ETH-TRY, SOL-TRY",TRY,None,crypto
5,BTC,8,3,11,"BTC-AUD, BTC-AED, BTC-BRL",BTC,None,crypto
6,ETH,9,2,11,"ETH-AUD, ETH-AED, ETH-BRL",ETH,None,crypto
7,SOL,9,1,10,"SOL-AUD, SOL-AED, SOL-BRL",SOL,None,crypto
8,BRL,0,10,10,"BTC-BRL, ETH-BRL, SOL-BRL",BRL,None,crypto
9,AUD,0,9,9,"BTC-AUD, ETH-AUD, SOL-AUD",AUD,None,crypto


## Match codes to local assets

Strategy, in order:
1. **`already_mapped`** — there's already an active `provider_asset` row for this code on this provider
2. **`exact_symbol`** — `Asset.symbol == code` (case-sensitive)
3. **`exact_name`** — `Asset.name == code` (case-sensitive)
4. **`ci_symbol`** — case-insensitive symbol match
5. **`ci_name`** — case-insensitive name match
6. **`fuzzy`** — difflib SequenceMatcher ≥ 0.85 on code or name (review-only, **not** auto-upserted)
7. **`unmatched`** — needs a manual override or a new `Asset` row

If multiple local assets match a code we record all candidate ids and flag `ambiguous = True`.

In [8]:
active = asset_df[asset_df["is_active"].fillna(True).astype(bool)].copy()

by_symbol = active.dropna(subset=["symbol"]).groupby("symbol")["id"].apply(list).to_dict()
by_name = active.dropna(subset=["name"]).groupby("name")["id"].apply(list).to_dict()
by_symbol_ci = (
    active.dropna(subset=["symbol"])
    .assign(_k=lambda d: d["symbol"].str.lower())
    .groupby("_k")["id"].apply(list).to_dict()
)
by_name_ci = (
    active.dropna(subset=["name"])
    .assign(_k=lambda d: d["name"].str.lower())
    .groupby("_k")["id"].apply(list).to_dict()
)

already_mapped_codes = set(
    existing_mappings.loc[existing_mappings["is_active"].fillna(True), "asset_code"]
)

asset_label = active.set_index("id").apply(
    lambda r: f"{r['symbol'] or r['name']} ({r['name']})", axis=1
).to_dict()


def match_code(code: str) -> dict[str, object]:
    if code in already_mapped_codes:
        ids = existing_mappings.loc[
            existing_mappings["asset_code"] == code, "asset_id"
        ].tolist()
        return {
            "match_type": "already_mapped",
            "asset_ids": ids,
            "asset_labels": [asset_label.get(i, str(i)) for i in ids],
        }
    for label, lookup, key in (
        ("exact_symbol", by_symbol, code),
        ("exact_name", by_name, code),
        ("ci_symbol", by_symbol_ci, code.lower()),
        ("ci_name", by_name_ci, code.lower()),
    ):
        if key in lookup:
            ids = lookup[key]
            return {
                "match_type": label,
                "asset_ids": ids,
                "asset_labels": [asset_label.get(i, str(i)) for i in ids],
            }
    return {"match_type": "unmatched", "asset_ids": [], "asset_labels": []}


match_records = codes_df["code"].apply(match_code).tolist()
matches = pd.concat([codes_df.reset_index(drop=True), pd.DataFrame(match_records)], axis=1)
matches["asset_id"] = matches["asset_ids"].apply(
    lambda xs: xs[0] if len(xs) == 1 else None
)
matches["ambiguous"] = matches["asset_ids"].apply(lambda xs: len(xs) > 1)
matches["asset_labels_str"] = matches["asset_labels"].apply(", ".join)

matches["match_type"].value_counts(dropna=False)

match_type
unmatched       233
exact_symbol     72
Name: count, dtype: int64

In [9]:
fuzzy_targets: list[tuple[int, str, str]] = []
for _, a in active.iterrows():
    aid = int(a["id"])
    label = f"{a['symbol'] or a['name']} ({a['name']})"
    if isinstance(a["symbol"], str) and a["symbol"]:
        fuzzy_targets.append((aid, label, a["symbol"].lower()))
    if isinstance(a["name"], str) and a["name"]:
        fuzzy_targets.append((aid, label, a["name"].lower()))


def fuzzy_best(query: str) -> tuple[int | None, str, float]:
    q = (query or "").lower().strip()
    if not q:
        return (None, "", 0.0)
    best = (None, "", 0.0)
    for aid, label, target in fuzzy_targets:
        score = difflib.SequenceMatcher(None, q, target).ratio()
        if score > best[2]:
            best = (aid, label, score)
    return best


matches["fuzzy_score"] = float("nan")
matches["fuzzy_query"] = ""

unmatched_idx = matches.index[matches["match_type"] == "unmatched"]
for idx in unmatched_idx:
    code = matches.at[idx, "code"]
    name_val = matches.at[idx, "provider_name"]
    name = name_val if isinstance(name_val, str) else None

    candidates: list[tuple[int | None, str, float, str]] = []
    if name:
        aid, label, score = fuzzy_best(name)
        candidates.append((aid, label, score, name))
    aid, label, score = fuzzy_best(code)
    candidates.append((aid, label, score, code))

    aid, label, score, query = max(candidates, key=lambda x: x[2])
    matches.at[idx, "fuzzy_score"] = score
    matches.at[idx, "fuzzy_query"] = query
    if score >= FUZZY_THRESHOLD and aid is not None:
        matches.at[idx, "match_type"] = "fuzzy"
        matches.at[idx, "asset_id"] = aid
        matches.at[idx, "asset_ids"] = [aid]
        matches.at[idx, "asset_labels"] = [label]
        matches.at[idx, "asset_labels_str"] = label

print(
    f"Fuzzy promoted {(matches['match_type'] == 'fuzzy').sum()} of "
    f"{len(unmatched_idx)} previously unmatched codes (threshold={FUZZY_THRESHOLD})"
)
matches["match_type"].value_counts(dropna=False)

Fuzzy promoted 6 of 233 previously unmatched codes (threshold=0.85)


match_type
unmatched       227
exact_symbol     72
fuzzy             6
Name: count, dtype: int64

## Review table

Sorted by match type then by total product volume. Skim the `unmatched` block first — those need a new `Asset` row or a manual override. The `asset_provider_codes` column shows how the matched local asset is coded by every other provider already in the DB.

In [10]:
matches["asset_description"] = matches["asset_id"].map(
    lambda x: asset_description_by_id.get(int(x)) if pd.notna(x) else None
)
matches["asset_provider_codes"] = matches["asset_id"].map(
    lambda x: provider_codes_by_asset.get(int(x), "") if pd.notna(x) else ""
)

match_order = pd.Categorical(
    matches["match_type"],
    categories=[
        "unmatched",
        "fuzzy",
        "ambiguous",
        "ci_name",
        "ci_symbol",
        "exact_name",
        "exact_symbol",
        "already_mapped",
    ],
    ordered=True,
)
matches_sorted = matches.assign(_order=match_order).sort_values(
    ["_order", "fuzzy_score", "total_products"],
    ascending=[True, False, False],
    na_position="last",
).drop(columns="_order")

review_cols = [
    "code",
    "provider_name",
    "provider_type",
    "provider_description",
    "match_type",
    "fuzzy_score",
    "fuzzy_query",
    "asset_id",
    "asset_labels_str",
    "asset_description",
    "asset_provider_codes",
    "ambiguous",
    "base_count",
    "quote_count",
    "total_products",
    "example_products",
]
review_df = matches_sorted[review_cols].rename(
    columns={"asset_labels_str": "asset_labels"}
)
review_path = f"review_{PROVIDER_NAME.lower()}.csv"
review_df.to_csv(review_path, index=False)
print(f"Wrote {len(review_df):,} rows to {review_path}")
review_df

Wrote 305 rows to review_okx.csv


,code,provider_name,provider_type,provider_description,match_type,fuzzy_score,fuzzy_query,asset_id,asset_labels,asset_description,asset_provider_codes,ambiguous,base_count,quote_count,total_products,example_products
187,OL,OL,crypto,None,unmatched,0.80,OL,NaN,,None,,False,4,0,4,"OL-EUR, OL-USD, OL-USDT"
206,PEOPLE,PEOPLE,crypto,None,unmatched,0.80,PEOPLE,NaN,,None,,False,4,0,4,"PEOPLE-EUR, PEOPLE-USD, PEOPLE-USDT"
245,AR,AR,crypto,None,unmatched,0.80,AR,NaN,,None,,False,4,0,4,"AR-EUR, AR-USD, AR-USDT"
277,SD,SD,crypto,None,unmatched,0.80,SD,NaN,,None,,False,2,0,2,"SD-USD, SD-USDT"
25,BARD,BARD,crypto,None,unmatched,0.75,BARD,NaN,,None,,False,5,0,5,"BARD-EUR, BARD-TRY, BARD-USD"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,SKY,SKY,crypto,None,exact_symbol,NaN,,57.0,SKY (SKY),Sky,"Coinbase:SKY, Kraken:SKY",False,4,0,4,"SKY-EUR, SKY-USD, SKY-USDT"
260,A,A,crypto,None,exact_symbol,NaN,,83.0,A (A),Vaulta,Kraken:A,False,4,0,4,"A-EUR, A-USD, A-USDT"
269,PAXG,PAXG,crypto,None,exact_symbol,NaN,,76.0,PAXG (PAXG),PAX Gold,"Coinbase:PAXG, Kraken:PAXG",False,3,0,3,"PAXG-TRY, PAXG-USD, PAXG-USDT"
297,RLUSD,RLUSD,crypto,None,exact_symbol,NaN,,87.0,RLUSD (RLUSD),Ripple USD,,False,1,0,1,RLUSD-USDT


## Local assets with no provider product

Inverse view: which active assets in our local `asset` table don't appear in any of the provider's products? These won't get a `provider_asset` row from this run.

In [11]:
provider_codes_set = set(codes_df["code"])
provider_codes_lower = {c.lower() for c in provider_codes_set}

matched_asset_ids = set(
    int(x) for x in matches["asset_ids"].explode().dropna().tolist()
)

local_missing = active[~active["id"].isin(matched_asset_ids)].copy()
local_missing["symbol_lower"] = local_missing["symbol"].str.lower()
local_missing["provider_has_symbol_ci"] = local_missing["symbol_lower"].isin(
    provider_codes_lower
)

local_missing = local_missing[
    ["id", "symbol", "name", "provider_has_symbol_ci"]
].sort_values(["provider_has_symbol_ci", "symbol"], ascending=[False, True])

missing_path = f"local_assets_no_{PROVIDER_NAME.lower()}.csv"
local_missing.to_csv(missing_path, index=False)
collisions = int(local_missing["provider_has_symbol_ci"].sum())
print(
    f"{len(local_missing):,} active local assets are not matched to a {PROVIDER_NAME} "
    f"product ({collisions} have a symbol that DOES exist on {PROVIDER_NAME} but matched "
    "a different local asset — check those for ticker collisions)"
)
local_missing

10 active local assets are not matched to a OKX product (0 have a symbol that DOES exist on OKX but matched a different local asset — check those for ticker collisions)


,id,symbol,name,provider_has_symbol_ci
67,75,CAKE,CAKE,False
14,28,DAI,DAI,False
69,77,FARTCOIN,FARTCOIN,False
76,84,JASMY,JASMY,False
35,42,KAS,KAS,False
26,33,MNT,MNT,False
54,61,QNT,QNT,False
55,62,SPX,SPX,False
28,35,TAO,TAO,False
23,30,XMR,XMR,False


## Sanity check: provider price vs reference provider close

For each matched asset with a USD or USDT pair on this provider, look up the latest USD close from the reference provider (default Kraken) and flag pairs that differ by more than `PRICE_DIFF_PCT_FLAG` percent. Big divergences usually mean a ticker collision — the code matched to the wrong local asset.

**Note:** this requires the connector to populate a `price` column on its products. Coinbase pulls `price` directly from `/products`; OKX pulls it from `/market/tickers` and merges by `instId`. Kraken's `AssetPairs` endpoint omits price and the per-pair `Ticker` endpoint requires N requests, so for Kraken this cell is skipped — fall back to the post-upsert price check once market data has flowed through the loader.

In [12]:
ref_provider = provider_df[provider_df["name"] == REFERENCE_PROVIDER_NAME]
ref_provider_id = (
    int(ref_provider["id"].iloc[0]) if not ref_provider.empty else None
)

usd_asset = active[active["symbol"] == "USD"]
usd_asset_id = int(usd_asset["id"].iloc[0]) if not usd_asset.empty else None

matched_for_price = matches[
    matches["match_type"].isin(
        ["exact_symbol", "exact_name", "ci_symbol", "ci_name", "already_mapped"]
    )
    & matches["asset_id"].notna()
    & ~matches["ambiguous"]
][["code", "asset_id", "provider_name", "asset_labels_str"]]
matched_asset_ids_list = matched_for_price["asset_id"].astype(int).tolist()

# Provider's current prices: only available if connector exposed a `price` column.
# Use base/USD or base/USDT as the price-bearing pair.
if "price" in live_df.columns:
    base_col = "base_code"
    quote_col = "quote_code"
    for quote in ("USD", "USDT"):
        cand = live_df[live_df[quote_col] == quote][[base_col, "price", "product_id"]].copy()
        if not cand.empty:
            cand["price"] = pd.to_numeric(cand["price"], errors="coerce")
            cand = cand.rename(
                columns={
                    base_col: "code",
                    "price": "provider_price",
                    "product_id": "provider_product_id",
                }
            )
            break
    has_price = True
else:
    has_price = False
    cand = pd.DataFrame()

if (
    not has_price
    or ref_provider_id is None
    or usd_asset_id is None
    or not matched_asset_ids_list
    or cand.empty
):
    print(
        f"Skipping price check: has_price={has_price}, ref_provider_id={ref_provider_id}, "
        f"usd_asset_id={usd_asset_id}, matched_asset_ids={len(matched_asset_ids_list)}, "
        f"price_pairs={0 if cand is None or cand.empty else len(cand)}"
    )
    price_check = pd.DataFrame()
else:
    cutoff = dt.datetime.now(dt.timezone.utc) - dt.timedelta(days=REFERENCE_LOOKBACK_DAYS)
    m = models.ProviderAssetMarket
    latest_stmt = (
        select(m.to_asset_id, m.timestamp, m.close)
        .where(m.timestamp >= cutoff)
        .where(m.provider_id == ref_provider_id)
        .where(m.from_asset_id == usd_asset_id)
        .where(m.to_asset_id.in_(matched_asset_ids_list))
        .order_by(m.to_asset_id, m.timestamp.desc())
        .distinct(m.to_asset_id)
    )
    latest_ref = pd.read_sql(latest_stmt, engine).rename(
        columns={
            "to_asset_id": "asset_id",
            "close": "reference_close",
            "timestamp": "reference_timestamp",
        }
    )

    price_check = (
        matched_for_price.merge(cand, on="code", how="inner")
        .merge(latest_ref, on="asset_id", how="left")
    )
    price_check["pct_diff"] = (
        (price_check["provider_price"] - price_check["reference_close"])
        / price_check["reference_close"]
        * 100
    )
    price_check["abs_pct_diff"] = price_check["pct_diff"].abs()
    price_check = price_check.sort_values(
        "abs_pct_diff", ascending=False, na_position="first"
    )

    cols = [
        "code",
        "provider_name",
        "asset_id",
        "asset_labels_str",
        "provider_product_id",
        "provider_price",
        "reference_close",
        "reference_timestamp",
        "pct_diff",
    ]
    price_check = price_check[cols].rename(columns={"asset_labels_str": "asset_labels"})
    price_path = f"price_check_{PROVIDER_NAME.lower()}.csv"
    price_check.to_csv(price_path, index=False)

    n_no_ref = price_check["reference_close"].isna().sum()
    n_big = (price_check["pct_diff"].abs() > PRICE_DIFF_PCT_FLAG).sum()
    print(
        f"{len(price_check)} matched assets have a {PROVIDER_NAME} USD/USDT pair; "
        f"{n_no_ref} have no {REFERENCE_PROVIDER_NAME} close in the last "
        f"{REFERENCE_LOOKBACK_DAYS}d; {n_big} differ by >{PRICE_DIFF_PCT_FLAG}%"
    )

price_check

68 matched assets have a OKX USD/USDT pair; 0 have no Kraken close in the last 7d; 0 differ by >5.0%


,code,provider_name,asset_id,asset_labels,provider_product_id,provider_price,reference_close,reference_timestamp,pct_diff
62,S,S,71.0,S (S),S-USD,0.04304,0.04500,2026-05-01 19:04:00,-4.355556
52,IMX,IMX,67.0,IMX (IMX),IMX-USD,0.16610,0.16990,2026-05-01 19:02:00,-2.236610
9,RENDER,RENDER,46.0,RENDER (RENDER),RENDER-USD,1.68200,1.72000,2026-05-01 19:03:00,-2.209302
60,PENDLE,PENDLE,73.0,PENDLE (PENDLE),PENDLE-USD,1.56500,1.59600,2026-05-01 19:03:00,-1.942356
47,VIRTUAL,VIRTUAL,82.0,VIRTUAL (VIRTUAL),VIRTUAL-USD,0.69800,0.71020,2026-05-01 19:04:00,-1.717826
...,...,...,...,...,...,...,...,...,...
0,USDT,USDT,18.0,USDT (USDT),USDT-USD,0.99990,0.99981,2026-05-01 19:04:00,0.009002
28,LTC,LTC,23.0,LTC (LTC),LTC-USD,55.96000,55.96000,2026-05-01 19:04:00,0.000000
13,STX,STX,64.0,STX (STX),STX-USD,0.22410,0.22410,2026-05-01 19:04:00,0.000000
12,SUI,SUI,14.0,SUI (SUI),SUI-USD,0.92470,0.92470,2026-05-01 19:04:00,0.000000


## Upsert provider_asset rows for matched, non-ambiguous codes

Writes one `provider_asset` row per `exact_symbol` / `exact_name` / `ci_symbol` / `ci_name` match that resolved to a single local asset and isn't already mapped. Uses Postgres `INSERT ... ON CONFLICT (date, provider_id, asset_id) DO UPDATE` so re-running is idempotent on a given date.

**Skim the review CSV first** — anything wrong should be corrected by adding/fixing the corresponding `Asset` row before running this cell. Fuzzy matches are intentionally excluded; promote them by adding `"fuzzy"` to the filter once you trust the score.

In [14]:
insertable = matches_sorted[
    matches_sorted["match_type"].isin(
        ["exact_symbol", "exact_name", "ci_symbol", "ci_name"]
    )
    & matches_sorted["asset_id"].notna()
    & ~matches_sorted["ambiguous"]
].copy()

today = dt.date.today() - dt.timedelta(days=365 * 2)  # use yesterday's date to avoid timezone issues
rows = [
    {
        "date": today,
        "provider_id": this_provider_id,
        "asset_id": int(row["asset_id"]),
        "asset_code": row["code"],
        "is_active": True,
    }
    for _, row in insertable.iterrows()
]

if not rows:
    print("No insertable rows.")
else:
    stmt = pg_insert(models.ProviderAsset).values(rows)
    stmt = stmt.on_conflict_do_update(
        index_elements=["date", "provider_id", "asset_id"],
        set_={
            "asset_code": stmt.excluded.asset_code,
            "is_active": stmt.excluded.is_active,
        },
    )
    with engine.begin() as conn:
        conn.execute(stmt)
    print(f"Upserted {len(rows)} provider_asset rows for {PROVIDER_NAME} on {today}")

Upserted 72 provider_asset rows for OKX on 2024-05-01
